<a href="https://colab.research.google.com/github/Srushanth/RAG/blob/modular-rag/Modular-RAG/notebooks/modular-rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧩 Modular RAG with LlamaIndex Workflows

This notebook explores **Modular RAG**. Instead of relying on rigid, high-level abstractions like `index.as_query_engine()`, we break down our RAG architecture into explicit, event-driven modules.

LlamaIndex recently transitioned from `QueryPipelines` to **`Workflows`**. Workflows are built on an event-driven architecture that allows for highly customizable, scalable, and complex routing (including loops for agentic logic) that was difficult to achieve in older DAG-based pipelines.

We will build a workflow consisting of:
1.  **Ingestion & Indexing:** Document loading, chunking, and embedding.
2.  **Retrieval Step:** An event listener that fetches context from the vector store.
3.  **Synthesis Step:** An event listener that formats the prompt and calls the LLM.

---
## 1. Setup & Configuration

In [1]:
! pip install "llama-index-core>=0.14.21" "llama-index-embeddings-huggingface>=0.7.0" "llama-index-llms-google-genai>=0.9.1" "sentence-transformers>=4.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 18.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, w

In [1]:
import os
import nest_asyncio

nest_asyncio.apply()

In [2]:
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [3]:
import getpass

In [7]:
# Set your Gemini API Key
os.environ["GEMINI_API_KEY"] = getpass.getpass("YOUR_GEMINI_API_KEY: ")

YOUR_GEMINI_API_KEY: ··········


In [8]:
# We use Gemini 1.5 Flash for rapid synthesis and a local HuggingFace embedding model
Settings.llm = GoogleGenAI(model="gemini-3-flash-preview")
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", device="cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
print(f"LLM: {Settings.llm.model}")
print(f"Embed model: {Settings.embed_model.model_name}")

LLM: gemini-3-flash-preview
Embed model: BAAI/bge-small-en-v1.5


---
## 2. The Ingestion & Indexing Module

First, we explicitly define our document parsing and node extraction. Instead of a single call to build the index, we separate the parsing logic from the vector store construction. This allows us to apply custom node parsers or metadata extractors if needed.

*Note: For this example, ensure you have a `data/` directory with a document (e.g., an Apple Environmental Progress Report).*

In [11]:
import os
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter

In [12]:
print("Loading documents...")
documents = SimpleDirectoryReader("data").load_data()

Loading documents...


In [13]:
# Explicitly define our chunking strategy
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

In [14]:
print("Extracting nodes...")
nodes = splitter.get_nodes_from_documents(documents)

Extracting nodes...


In [15]:
print(f"Extracted {len(nodes)} nodes. Building index...")
index = VectorStoreIndex(nodes, show_progress=True)

Extracted 28464 nodes. Building index...


Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

...

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1840 [00:00<?, ?it/s]

---
## 3. Defining the Modular RAG Workflow

In LlamaIndex, a **Workflow** is defined by creating a class that inherits from `Workflow`, and decorating asynchronous methods with `@step`.

Communication between steps is handled by **Events**.
1. `StartEvent`: A built-in event containing the user's initial input.
2. `RetrievalEvent`: A custom event we create to pass retrieved nodes from the Retriever step to the Synthesizer step.
3. `StopEvent`: A built-in event that terminates the workflow and returns the final output.

In [16]:
from llama_index.core.workflow import (
    Workflow,
    StartEvent,
    StopEvent,
    step,
    Event
)
from llama_index.core import PromptTemplate

In [17]:
# 1. Define Custom Events
class RetrievalEvent(Event):
    """Event containing the retrieved nodes and the original query."""
    nodes: list
    query: str

In [18]:
# 2. Define the Workflow
class ModularRAGWorkflow(Workflow):
    def __init__(self, index, llm, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.index = index
        self.llm = llm

        # Define our Prompt Template module
        prompt_str = (
            "Context information is below.\n"
            "---------------------\n"
            "{context_str}\n"
            "---------------------\n"
            "Given the context information, answer the user's query.\n"
            "Query: {query_str}\n"
            "Answer: "
        )
        self.prompt_tmpl = PromptTemplate(prompt_str)

    @step
    async def retrieve(self, ev: StartEvent) -> RetrievalEvent:
        """Module 1: The Retrieval Step. Listens for StartEvent."""
        query = ev.query
        print(f"[Step 1: Retriever] Fetching context for: '{query}'")

        retriever = self.index.as_retriever(similarity_top_k=3)
        nodes = await retriever.aretrieve(query)

        # Emit a RetrievalEvent to trigger the next step
        return RetrievalEvent(nodes=nodes, query=query)

    @step
    async def synthesize(self, ev: RetrievalEvent) -> StopEvent:
        """Module 2: The Synthesis Step. Listens for RetrievalEvent."""
        print(f"[Step 2: Synthesizer] Formatting prompt and calling LLM...")

        # Format the retrieved nodes into a single string
        context_str = "\n\n".join([n.get_content() for n in ev.nodes])

        # Format the prompt using our template
        formatted_prompt = self.prompt_tmpl.format(
            context_str=context_str,
            query_str=ev.query
        )

        # Call the LLM Module
        response = await self.llm.acomplete(formatted_prompt)

        # Return a StopEvent with the final result
        return StopEvent(result=str(response))

---
## 4. Execution

To run the workflow, we instantiate our custom class and call the async `.run()` method, passing the initial query.

In [19]:
# Instantiate the modular workflow
workflow = ModularRAGWorkflow(index=index, llm=Settings.llm)

In [22]:
query = "What are the main topics discussed in the docment?"

print(f"\nExecuting Workflow for query: '{query}'\n")


Executing Workflow for query: 'What are the main topics discussed in the docment?'



In [23]:
# Run the workflow (we must await it since workflows are natively asynchronous)
result = await workflow.run(query=query)

print("\n--------------------------------")
print("Final Output:")
print("--------------------------------")
print(result)

[Step 1: Retriever] Fetching context for: 'What are the main topics discussed in the docment?'
[Step 2: Synthesizer] Formatting prompt and calling LLM...

--------------------------------
Final Output:
--------------------------------
Based on the metadata provided in the context, the main topic of the document is the **Assurance of Sustainability Reports**.

Specifically, the document is a **Template Assurance Statement (Medium)**. It appears to be a formal template used by an organization (indicated by the creator "BV User," likely referring to Bureau Veritas) to provide assurance services for sustainability reporting.
